# 🏗️ DeepSeek V3 전체 모델 구조와 추론 흐름

이 노트북에서는 DeepSeek V3의 **전체 모델 구조**를 살펴보고, **추론 과정**에서 데이터가 어떻게 흐르는지 학습합니다.

**참고 자료:**
- 논문: https://arxiv.org/pdf/2412.19437

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

print("✅ 라이브러리 로드 완료")

✅ 라이브러리 로드 완료


## 1. DeepSeek V3 설정

### 주요 스펙
| 구성 요소 | 값 | 설명 |
|-----------|-----|------|
| vocab_size | 102,400 | 토큰 어휘 크기 |
| hidden_size | 7,168 | 히든 차원 |
| num_hidden_layers | 61 | 디코더 레이어 수 |
| num_attention_heads | 128 | 어텐션 헤드 수 |
| kv_lora_rank | 512 | KV 압축 차원 |
| n_routed_experts | 256 | 라우팅 전문가 수 |
| num_experts_per_tok | 8 | 토큰당 활성화 전문가 |

In [2]:
@dataclass
class DeepSeekV3Config:
    # 기본 설정
    vocab_size: int = 102400
    hidden_size: int = 7168
    num_hidden_layers: int = 61
    num_attention_heads: int = 128
    
    # MLA 설정
    kv_lora_rank: int = 512
    q_lora_rank: int = 1536
    qk_nope_head_dim: int = 128
    qk_rope_head_dim: int = 64
    v_head_dim: int = 128
    
    # MoE 설정
    n_routed_experts: int = 256
    n_shared_experts: int = 2
    num_experts_per_tok: int = 8
    moe_intermediate_size: int = 2048
    
    # 기타
    intermediate_size: int = 18432
    max_position_embeddings: int = 163840
    first_k_dense_replace: int = 3

config = DeepSeekV3Config()

print("📊 DeepSeek V3 설정")
print("=" * 50)
for field, value in vars(config).items():
    print(f"{field:>25}: {value:>10,}" if isinstance(value, int) else f"{field:>25}: {value}")

📊 DeepSeek V3 설정
               vocab_size:    102,400
              hidden_size:      7,168
        num_hidden_layers:         61
      num_attention_heads:        128
             kv_lora_rank:        512
              q_lora_rank:      1,536
         qk_nope_head_dim:        128
         qk_rope_head_dim:         64
               v_head_dim:        128
         n_routed_experts:        256
         n_shared_experts:          2
      num_experts_per_tok:          8
    moe_intermediate_size:      2,048
        intermediate_size:     18,432
  max_position_embeddings:    163,840
    first_k_dense_replace:          3


## 2. 전체 아키텍처 시각화

In [3]:
print("""
┌─────────────────────────────────────────────────────────────┐
│                   DeepSeek V3 Architecture                  │
│               총 671B 파라미터 / 토큰당 37B 활성화            │
└─────────────────────────────────────────────────────────────┘

                      Input Tokens
                          │
                          ▼
                  ┌───────────────┐
                  │Token Embedding│
                  └───────┬───────┘
                          │
    ┌─────────────────────┼─────────────────────┐
    │                     │                     │
    │  ┌──────────────────────────────────────┐ │
    │  │        Decoder Layer 0-2             │ │
    │  │           (Dense MLP)                │ │
    │  │  RMSNorm → MLA → Add Residual        │ │
    │  │  RMSNorm → MLP → Add Residual        │ │
    │  └──────────────────────────────────────┘ │
    │                     │                     │
    │  ┌──────────────────────────────────────┐ │
    │  │        Decoder Layer 3-60            │ │
    │  │            (MoE)                     │ │
    │  │  RMSNorm → MLA → Add Residual        │ │
    │  │  RMSNorm → MoE → Add Residual        │ │
    │  │  (256 experts, 8 active per token)   │ │
    │  └──────────────────────────────────────┘ │
    │                     │                     │
    │               × 61 layers                 │
    └─────────────────────┼─────────────────────┘
                          │
                  ┌───────────────┐
                  │ Final RMSNorm │
                  └───────┬───────┘
                          │
                  ┌───────────────┐
                  │   LM Head     │
                  └───────┬───────┘
                          │
                          ▼
                    Output Logits
""")

print("\n💡 핵심 포인트:")
print("   - 처음 3개 레이어는 Dense MLP")
print("   - 나머지 58개 레이어는 MoE")
print("   - 모든 레이어에서 MLA 사용")


┌─────────────────────────────────────────────────────────────┐
│                   DeepSeek V3 Architecture                  │
│               총 671B 파라미터 / 토큰당 37B 활성화            │
└─────────────────────────────────────────────────────────────┘

                      Input Tokens
                          │
                          ▼
                  ┌───────────────┐
                  │Token Embedding│
                  └───────┬───────┘
                          │
    ┌─────────────────────┼─────────────────────┐
    │                     │                     │
    │  ┌──────────────────────────────────────┐ │
    │  │        Decoder Layer 0-2             │ │
    │  │           (Dense MLP)                │ │
    │  │  RMSNorm → MLA → Add Residual        │ │
    │  │  RMSNorm → MLP → Add Residual        │ │
    │  └──────────────────────────────────────┘ │
    │                     │                     │
    │  ┌──────────────────────────────────────┐ │
    │  │        Decoder 

## 📝 이해도 테스트 1: 파라미터 계산

In [4]:
# 파라미터 수 계산

# Embedding
embed_params = config.vocab_size * config.hidden_size

# Dense MLP (per layer)
dense_mlp_params = 3 * config.hidden_size * config.intermediate_size

# MoE (per layer)
expert_params = 3 * config.hidden_size * config.moe_intermediate_size
moe_params = (config.n_routed_experts * expert_params + 
              config.n_shared_experts * expert_params +
              config.n_routed_experts * config.hidden_size)  # router

# LM Head
lm_head_params = config.hidden_size * config.vocab_size

print("📊 파라미터 분석")
print("=" * 60)
print(f"Token Embedding:           {embed_params:>15,}")
print(f"Dense MLP (per layer):     {dense_mlp_params:>15,}")
print(f"MoE (per layer):           {moe_params:>15,}")
print(f"LM Head:                   {lm_head_params:>15,}")

# 총계 추정 (간소화)
dense_layers = config.first_k_dense_replace
moe_layers = config.num_hidden_layers - dense_layers

total_estimate = (embed_params + 
                  dense_layers * dense_mlp_params +
                  moe_layers * moe_params +
                  lm_head_params)

print(f"\n총 파라미터 추정치: {total_estimate/1e9:.1f}B")
print("(실제값: 671B - MLA 파라미터 등 미포함)")

📊 파라미터 분석
Token Embedding:               734,003,200
Dense MLP (per layer):         396,361,728
MoE (per layer):            11,364,204,544
LM Head:                       734,003,200

총 파라미터 추정치: 661.8B
(실제값: 671B - MLA 파라미터 등 미포함)


## 3. 추론 흐름

### Autoregressive Generation 과정

**Step 1: Prefill Phase (입력 처리)**
- 전체 입력 시퀀스를 한 번에 처리
- KV 캐시 저장

**Step 2: Decode Phase (토큰 생성)**
- 새 토큰 하나씩 생성
- KV 캐시 활용으로 효율적 처리

In [5]:
print("""
🔄 Autoregressive Generation 과정
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Step 1: Prefill Phase
─────────────────────

   Input: "DeepSeek V3는"
   
   ┌────────────────────────────────────────────────────────┐
   │ 1. Tokenization                                        │
   │    "DeepSeek V3는" → [token_1, token_2, token_3, ...]  │
   │                                                        │
   │ 2. Forward through all layers                          │
   │    - MLA: 모든 토큰 간 어텐션                            │
   │    - KV Cache 저장 (c_KV, k_rope)                      │
   │    - MoE: 각 토큰별 Top-8 전문가                        │
   │                                                        │
   │ 3. LM Head → Sampling → next_token                     │
   └────────────────────────────────────────────────────────┘

Step 2: Decode Phase (반복)
─────────────────────────

   ┌────────────────────────────────────────────────────────┐
   │ 1. 새 토큰 임베딩                                       │
   │                                                        │
   │ 2. Forward with KV Cache                               │
   │    - Query: 새 토큰만 계산                              │
   │    - Key, Value: 캐시에서 가져옴                        │
   │                                                        │
   │ 3. LM Head → Sampling → next_token                     │
   │                                                        │
   │ 4. KV Cache 업데이트                                   │
   └────────────────────────────────────────────────────────┘
   
   → EOS 토큰이 나올 때까지 반복

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print("💡 핵심: KV 캐시 덕분에 Decode Phase가 효율적!")


🔄 Autoregressive Generation 과정
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Step 1: Prefill Phase
─────────────────────

   Input: "DeepSeek V3는"
   
   ┌────────────────────────────────────────────────────────┐
   │ 1. Tokenization                                        │
   │    "DeepSeek V3는" → [token_1, token_2, token_3, ...]  │
   │                                                        │
   │ 2. Forward through all layers                          │
   │    - MLA: 모든 토큰 간 어텐션                            │
   │    - KV Cache 저장 (c_KV, k_rope)                      │
   │    - MoE: 각 토큰별 Top-8 전문가                        │
   │                                                        │
   │ 3. LM Head → Sampling → next_token                     │
   └────────────────────────────────────────────────────────┘

Step 2: Decode Phase (반복)
─────────────────────────

   ┌────────────────────────────────────────────────────────┐
   │ 1. 새 토큰 임베딩                               

## 4. Multi-Token Prediction (MTP)

### 개념
- 기존: 한 번에 하나의 다음 토큰 예측
- MTP: 한 번에 여러 개의 토큰 예측

### 장점
1. 학습 효율성 향상
2. Speculative Decoding으로 추론 가속화

In [6]:
print("""
📊 Multi-Token Prediction (MTP)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

기존 Next-Token Prediction:
  Input: "The capital of France is"
  예측: [Paris]  (1개)

Multi-Token Prediction:
  Input: "The capital of France is"
  예측: [Paris, ., It, is]  (4개 동시!)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Speculative Decoding 과정:

  Step 1: Draft (MTP 헤드로 여러 토큰 예측)
          → [Paris, ., It, is]
  
  Step 2: Verify (메인 모델로 검증)
          → [Paris ✅, . ✅, It ❌, ...]
  
  Step 3: Accept
          → "Paris ." 출력 (한 번에 2개 토큰!)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print("💡 핵심: 예측 + 검증으로 추론 속도 향상!")


📊 Multi-Token Prediction (MTP)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

기존 Next-Token Prediction:
  Input: "The capital of France is"
  예측: [Paris]  (1개)

Multi-Token Prediction:
  Input: "The capital of France is"
  예측: [Paris, ., It, is]  (4개 동시!)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Speculative Decoding 과정:

  Step 1: Draft (MTP 헤드로 여러 토큰 예측)
          → [Paris, ., It, is]
  
  Step 2: Verify (메인 모델로 검증)
          → [Paris ✅, . ✅, It ❌, ...]
  
  Step 3: Accept
          → "Paris ." 출력 (한 번에 2개 토큰!)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

💡 핵심: 예측 + 검증으로 추론 속도 향상!


### MTP 구현 예시 (HuggingFace transformers 스타일)

DeepSeek V3의 MTP는 다음과 같은 핵심 구성요소로 이루어집니다:

1. **MTP Embedding Layer**: 각 depth에서 토큰 임베딩을 처리
2. **MTP Transformer Block**: 이전 표현과 미래 토큰 임베딩을 결합하여 처리  
3. **MTP Output Heads**: 각 depth에서 다음 토큰을 예측

참고: [HuggingFace DeepSeek V3 구현](https://github.com/huggingface/transformers/blob/main/src/transformers/models/deepseek_v3/modeling_deepseek_v3.py)


In [7]:
# MTP 구현 예시 (HuggingFace transformers 스타일 참조)
# 참고: https://github.com/huggingface/transformers/blob/main/src/transformers/models/deepseek_v3/modeling_deepseek_v3.py

class DeepSeekV3RMSNorm(nn.Module):
    """RMSNorm - DeepSeek V3에서 사용하는 정규화 레이어"""
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        return self.weight * hidden_states.to(input_dtype)


class DeepSeekV3MLP(nn.Module):
    """MLP 레이어 - Gate-Up-Down 구조"""
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        self.act_fn = nn.SiLU()

    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))


class MTPBlock(nn.Module):
    """
    Multi-Token Prediction Block
    
    각 MTP depth에서 사용되는 Transformer 블록입니다.
    이전 hidden states와 다음 토큰의 임베딩을 결합하여 처리합니다.
    
    수식:
        h'_i = Concat(h_{i-1}, Embed(t_i)) → Linear → Transformer Block
    """
    def __init__(self, hidden_size, intermediate_size, num_heads=8, eps=1e-6):
        super().__init__()
        self.hidden_size = hidden_size
        
        # 이전 hidden state와 임베딩을 결합하는 projection
        # 2 * hidden_size -> hidden_size
        self.enorm = DeepSeekV3RMSNorm(hidden_size, eps=eps)
        self.hnorm = DeepSeekV3RMSNorm(hidden_size, eps=eps)
        self.eh_proj = nn.Linear(hidden_size * 2, hidden_size, bias=False)
        
        # Self-attention (간소화된 버전)
        self.input_layernorm = DeepSeekV3RMSNorm(hidden_size, eps=eps)
        self.self_attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            batch_first=True
        )
        
        # MLP
        self.post_attention_layernorm = DeepSeekV3RMSNorm(hidden_size, eps=eps)
        self.mlp = DeepSeekV3MLP(hidden_size, intermediate_size)

    def forward(self, hidden_states, prev_embeddings):
        """
        Args:
            hidden_states: 이전 depth의 hidden states [batch, seq_len, hidden]
            prev_embeddings: 예측할 토큰의 임베딩 [batch, seq_len, hidden]
        """
        # 1. 이전 hidden state와 임베딩 결합
        # HuggingFace 구현: eh_proj(concat(enorm(embed), hnorm(hidden)))
        combined = torch.cat([
            self.enorm(prev_embeddings), 
            self.hnorm(hidden_states)
        ], dim=-1)
        hidden_states = self.eh_proj(combined)
        
        # 2. Self-attention with residual
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        hidden_states, _ = self.self_attn(
            hidden_states, hidden_states, hidden_states,
            need_weights=False
        )
        hidden_states = residual + hidden_states
        
        # 3. MLP with residual
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = residual + self.mlp(hidden_states)
        
        return hidden_states


print("✅ MTP 기본 블록 정의 완료")


✅ MTP 기본 블록 정의 완료


In [ ]:
class MultiTokenPredictionModule(nn.Module):
    """
    DeepSeek V3 Multi-Token Prediction (MTP) 모듈
    
    논문 설명:
    - 한 번에 여러 개의 미래 토큰을 예측
    - 각 depth k에서 k번째 미래 토큰을 예측
    - 학습 시: 더 풍부한 학습 신호 제공
    - 추론 시: Speculative Decoding으로 가속화
    
    HuggingFace 구현 참조:
    https://github.com/huggingface/transformers/blob/main/src/transformers/models/deepseek_v3/modeling_deepseek_v3.py
    """
    def __init__(
        self, 
        vocab_size=1024,
        hidden_size=256, 
        intermediate_size=512,
        num_mtp_heads=4,  # MTP depth (동시에 예측할 토큰 수)
        num_attention_heads=8,
        share_lm_head=True,  # LM head 공유 여부
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_mtp_heads = num_mtp_heads
        self.share_lm_head = share_lm_head
        
        # Token Embedding (메인 모델과 공유)
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        
        # MTP Blocks - 각 depth마다 하나의 Transformer 블록
        self.mtp_blocks = nn.ModuleList([
            MTPBlock(
                hidden_size=hidden_size,
                intermediate_size=intermediate_size,
                num_heads=num_attention_heads
            )
            for _ in range(num_mtp_heads)
        ])
        
        # Output Projection (LM Head)
        if share_lm_head:
            # 모든 depth에서 같은 LM head 사용
            self.shared_head = nn.Linear(hidden_size, vocab_size, bias=False)
            self.lm_heads = None
        else:
            # 각 depth마다 별도의 LM head
            self.lm_heads = nn.ModuleList([
                nn.Linear(hidden_size, vocab_size, bias=False)
                for _ in range(num_mtp_heads)
            ])
            self.shared_head = None
        
        # Final normalization for each depth
        self.norms = nn.ModuleList([
            DeepSeekV3RMSNorm(hidden_size)
            for _ in range(num_mtp_heads)
        ])

    def forward(self, main_hidden_states, input_ids=None, labels=None):
        """
        Args:
            main_hidden_states: 메인 모델의 마지막 hidden states [batch, seq_len, hidden]
            input_ids: 입력 토큰 IDs (학습 시 필요) [batch, seq_len]
            labels: 정답 토큰 IDs (학습 시 필요) [batch, seq_len]
            
        Returns:
            mtp_logits: 각 depth의 예측 logits 리스트
            mtp_loss: MTP 손실 (학습 시)
        """
        batch_size, seq_len, _ = main_hidden_states.shape
        
        mtp_logits = []
        mtp_loss = 0.0
        num_valid_depths = 0
        
        # 현재 hidden states (메인 모델 출력으로 시작)
        current_hidden = main_hidden_states
        
        for depth in range(self.num_mtp_heads):
            # 학습 시: 실제 토큰 임베딩 사용
            # 추론 시: 이전 예측 토큰 임베딩 사용
            if input_ids is not None:
                # 학습 모드: depth번째 미래 토큰의 임베딩
                # 위치 i에서 토큰 i+depth를 예측하므로, 임베딩은 i+depth 위치의 토큰
                if depth == 0:
                    # depth 0: 현재 위치의 토큰 임베딩 (이미 처리된 것)
                    token_embeds = self.embed_tokens(input_ids)
                else:
                    # depth > 0: 이전에 예측한 토큰 위치의 임베딩
                    # 실제로는 shift된 input_ids 사용
                    shifted_ids = F.pad(input_ids[:, depth:], (0, depth), value=0)
                    token_embeds = self.embed_tokens(shifted_ids)
            else:
                # 추론 모드: 이전 예측에서 얻은 토큰 사용
                # (실제 구현에서는 greedy/sampling으로 얻은 토큰)
                token_embeds = torch.zeros_like(current_hidden)
            
            # MTP Block 통과
            current_hidden = self.mtp_blocks[depth](current_hidden, token_embeds)
            
            # Normalize and project to vocab
            normalized = self.norms[depth](current_hidden)
            
            if self.share_lm_head:
                logits = self.shared_head(normalized)
            else:
                logits = self.lm_heads[depth](normalized)
            
            mtp_logits.append(logits)
            
            # 손실 계산 (학습 시)
            if labels is not None:
                # depth번째 미래 토큰 예측 손실
                # 위치 i에서 토큰 i+depth+1을 예측
                # logits: [batch, seq_len, vocab] -> 예측은 위치 0~(seq_len-1)
                # labels: [batch, seq_len] -> 타겟은 위치 (depth+1)~(seq_len-1)
                
                # 시퀀스 길이가 충분한지 확인
                if seq_len > depth + 1:
                    # logits에서 마지막 (depth+1)개 위치 제외 (타겟이 없으므로)
                    shift_logits = logits[:, :seq_len - depth - 1, :].contiguous()
                    # labels에서 앞의 (depth+1)개 위치 제외 (예측 대상이 아니므로)
                    shift_labels = labels[:, depth + 1:].contiguous()
                    
                    # 크기 맞추기 (더 작은 쪽에 맞춤)
                    min_len = min(shift_logits.size(1), shift_labels.size(1))
                    shift_logits = shift_logits[:, :min_len, :]
                    shift_labels = shift_labels[:, :min_len]
                    
                    loss = F.cross_entropy(
                        shift_logits.reshape(-1, self.vocab_size),
                        shift_labels.reshape(-1),
                        ignore_index=-100
                    )
                    mtp_loss = mtp_loss + loss
                    num_valid_depths += 1
        
        # 평균 MTP 손실
        if labels is not None and num_valid_depths > 0:
            mtp_loss = mtp_loss / num_valid_depths
        
        return mtp_logits, mtp_loss


print("✅ Multi-Token Prediction 모듈 정의 완료")


✅ Multi-Token Prediction 모듈 정의 완료


In [9]:
# MTP 모듈 테스트
print("🧪 MTP 모듈 테스트")
print("=" * 60)

# 설정
batch_size = 2
seq_len = 16
vocab_size = 1024
hidden_size = 256
num_mtp_heads = 4  # 4개 토큰 동시 예측

# MTP 모듈 생성
mtp_module = MultiTokenPredictionModule(
    vocab_size=vocab_size,
    hidden_size=hidden_size,
    intermediate_size=512,
    num_mtp_heads=num_mtp_heads,
    num_attention_heads=8,
    share_lm_head=True
)

# 가상의 메인 모델 출력
main_hidden_states = torch.randn(batch_size, seq_len, hidden_size)
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
labels = input_ids.clone()  # 학습용 라벨

# Forward pass
mtp_logits, mtp_loss = mtp_module(main_hidden_states, input_ids, labels)

print(f"\n📊 입력 형태:")
print(f"   - main_hidden_states: {main_hidden_states.shape}")
print(f"   - input_ids: {input_ids.shape}")

print(f"\n📊 출력 형태:")
for i, logits in enumerate(mtp_logits):
    print(f"   - MTP Head {i+1} logits: {logits.shape}")

print(f"\n📊 MTP 손실: {mtp_loss.item():.4f}")

# 파라미터 수 계산
total_params = sum(p.numel() for p in mtp_module.parameters())
print(f"\n📊 MTP 모듈 파라미터: {total_params:,}")


🧪 MTP 모듈 테스트


RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [ ]:
print("""
📊 MTP 아키텍처 시각화
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

                    Main Model Output
                          │
                    [hidden_states]
                          │
    ┌─────────────────────┴─────────────────────┐
    │                                           │
    ▼                                           │
┌─────────────────────────────────────────┐     │
│           MTP Block (Depth 0)           │     │
│                                         │     │
│  ┌─────────┐   ┌─────────┐             │     │
│  │ hidden  │ + │  embed  │ → concat    │     │
│  │ states  │   │(token_i)│             │     │
│  └────┬────┘   └────┬────┘             │     │
│       └──────┬──────┘                  │     │
│              ▼                         │     │
│       [eh_proj: 2d → d]                │     │
│              ▼                         │     │
│       [Transformer Block]              │     │
│              ▼                         │     │
│       [RMSNorm → LM Head]              │     │
└──────────────┬──────────────────────────┘     │
               │                                │
               ▼                                │
         logits_0 → 예측: token_{i+1}           │
               │                                │
    ┌──────────┴──────────┐                     │
    ▼                     │                     │
┌─────────────────────────────────────────┐     │
│           MTP Block (Depth 1)           │     │
│                                         │     │
│  hidden_0 + embed(token_{i+1}) → ...   │     │
└──────────────┬──────────────────────────┘     │
               │                                │
               ▼                                │
         logits_1 → 예측: token_{i+2}           │
               │                                │
               ▼                                │
              ...                               │
               │                                │
               ▼                                │
┌─────────────────────────────────────────┐     │
│           MTP Block (Depth K-1)         │     │
│                                         │     │
│  hidden_{k-2} + embed(token_{i+k-1})   │     │
└──────────────┬──────────────────────────┘     │
               │                                │
               ▼                                │
         logits_{k-1} → 예측: token_{i+k}       │

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print("💡 핵심: 각 depth에서 이전 hidden state와 토큰 임베딩을 결합하여 다음 토큰 예측!")


In [12]:
class SpeculativeDecoder:
    """
    MTP를 활용한 Speculative Decoding 구현
    
    과정:
    1. Draft: MTP 헤드로 여러 토큰 예측
    2. Verify: 메인 모델로 검증
    3. Accept: 검증된 토큰만 수락
    
    이점:
    - 한 번의 forward pass로 여러 토큰 생성 가능
    - 검증 단계에서 병렬 처리로 효율성 향상
    """
    def __init__(self, main_model, mtp_module, vocab_size):
        self.main_model = main_model  # 메인 Transformer 모델
        self.mtp_module = mtp_module  # MTP 모듈
        self.vocab_size = vocab_size
    
    def generate_draft(self, hidden_states, num_tokens=4):
        """
        MTP 헤드로 여러 토큰 draft 생성
        
        Args:
            hidden_states: 메인 모델의 hidden states
            num_tokens: 생성할 draft 토큰 수
            
        Returns:
            draft_tokens: 예측된 토큰들
            draft_logits: 각 토큰의 logits
        """
        draft_tokens = []
        draft_logits = []
        
        current_hidden = hidden_states
        
        for depth in range(min(num_tokens, self.mtp_module.num_mtp_heads)):
            # MTP 블록 통과
            if depth > 0 and len(draft_tokens) > 0:
                prev_embed = self.mtp_module.embed_tokens(draft_tokens[-1])
            else:
                prev_embed = torch.zeros_like(current_hidden)
            
            current_hidden = self.mtp_module.mtp_blocks[depth](current_hidden, prev_embed)
            
            # Logits 계산
            normalized = self.mtp_module.norms[depth](current_hidden)
            if self.mtp_module.share_lm_head:
                logits = self.mtp_module.shared_head(normalized)
            else:
                logits = self.mtp_module.lm_heads[depth](logits)
            
            # Greedy decoding (마지막 위치에서)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            
            draft_tokens.append(next_token)
            draft_logits.append(logits[:, -1, :])
        
        return draft_tokens, draft_logits
    
    def verify_and_accept(self, input_ids, draft_tokens, main_logits):
        """
        메인 모델로 draft 토큰 검증
        
        Args:
            input_ids: 원본 입력 토큰
            draft_tokens: MTP로 예측한 draft 토큰들
            main_logits: 메인 모델의 logits (draft 포함 전체 시퀀스)
            
        Returns:
            accepted_tokens: 수락된 토큰들
            num_accepted: 수락된 토큰 수
        """
        accepted_tokens = []
        
        for i, draft_token in enumerate(draft_tokens):
            # 메인 모델의 예측과 비교
            main_pred = main_logits[:, -len(draft_tokens)+i, :].argmax(dim=-1, keepdim=True)
            
            if (draft_token == main_pred).all():
                # Draft 토큰이 메인 모델 예측과 일치 → 수락
                accepted_tokens.append(draft_token)
            else:
                # 불일치 → 여기서 중단하고 메인 모델 예측 사용
                accepted_tokens.append(main_pred)
                break
        
        return accepted_tokens, len(accepted_tokens)


# Speculative Decoding 시뮬레이션
print("🚀 Speculative Decoding 시뮬레이션")
print("=" * 60)

# 가상의 시나리오
draft_tokens = ["Paris", ".", "It", "is"]
verification_results = ["✅", "✅", "❌", "-"]

print("\n📝 시나리오: 'The capital of France is' 다음 토큰 예측\n")

print("Step 1: Draft (MTP 헤드)")
print("─" * 40)
for i, (token, result) in enumerate(zip(draft_tokens, verification_results)):
    print(f"   Depth {i}: '{token}'")

print("\nStep 2: Verify (메인 모델)")
print("─" * 40)
for i, (token, result) in enumerate(zip(draft_tokens, verification_results)):
    if result == "-":
        print(f"   Position {i}: (검증 안 함)")
    else:
        print(f"   Position {i}: '{token}' {result}")

print("\nStep 3: Accept")
print("─" * 40)
accepted = [t for t, r in zip(draft_tokens, verification_results) if r == "✅"]
print(f"   수락된 토큰: {accepted}")
print(f"   한 번에 {len(accepted)}개 토큰 생성!")

print("\n" + "=" * 60)
print("💡 결과: 기존 1토큰/step → Speculative Decoding으로 2토큰/step")
print("   → 약 2배 속도 향상 (acceptance rate에 따라 다름)")


🚀 Speculative Decoding 시뮬레이션

📝 시나리오: 'The capital of France is' 다음 토큰 예측

Step 1: Draft (MTP 헤드)
────────────────────────────────────────
   Depth 0: 'Paris'
   Depth 1: '.'
   Depth 2: 'It'
   Depth 3: 'is'

Step 2: Verify (메인 모델)
────────────────────────────────────────
   Position 0: 'Paris' ✅
   Position 1: '.' ✅
   Position 2: 'It' ❌
   Position 3: (검증 안 함)

Step 3: Accept
────────────────────────────────────────
   수락된 토큰: ['Paris', '.']
   한 번에 2개 토큰 생성!

💡 결과: 기존 1토큰/step → Speculative Decoding으로 2토큰/step
   → 약 2배 속도 향상 (acceptance rate에 따라 다름)


In [ ]:
print("""
📊 MTP 학습 손실 계산 방식
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DeepSeek V3에서의 MTP 손실 계산:

  Total Loss = L_main + λ × L_MTP

  여기서:
    L_main = 기존 Next-Token Prediction 손실
    L_MTP  = Σ(k=1 to D) L_k / D  (각 depth 손실의 평균)
    λ      = MTP 손실 가중치 (논문에서는 0.3 사용)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

각 Depth에서의 손실:

  Depth 0: 위치 i에서 토큰 i+1 예측 (= 기존 NTP와 동일)
  Depth 1: 위치 i에서 토큰 i+2 예측
  Depth 2: 위치 i에서 토큰 i+3 예측
  ...
  Depth D-1: 위치 i에서 토큰 i+D 예측

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

예시 (D=4, 시퀀스 "The capital of France is Paris"):

  Position:  0     1       2     3       4    5
  Token:    The  capital  of  France   is  Paris
  
  Depth 0 예측: capital  of  France   is  Paris  [EOS]
  Depth 1 예측:    of  France   is  Paris  [EOS]  ...
  Depth 2 예측: France   is  Paris  [EOS]  ...   ...
  Depth 3 예측:    is  Paris  [EOS]  ...   ...   ...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print("💡 핵심: 더 먼 미래 토큰을 예측하면서 모델이 더 넓은 문맥을 학습!")


In [ ]:
# MTP 손실 계산 예시
print("🧮 MTP 손실 계산 예시")
print("=" * 60)

# 가상의 손실 값들
main_loss = 2.5
mtp_losses = [2.3, 2.7, 3.1, 3.5]  # depth 0, 1, 2, 3
mtp_weight = 0.3  # λ (논문 설정)

# 평균 MTP 손실
avg_mtp_loss = sum(mtp_losses) / len(mtp_losses)

# 총 손실
total_loss = main_loss + mtp_weight * avg_mtp_loss

print(f"\n📊 손실 구성:")
print(f"   Main Loss (L_main):     {main_loss:.2f}")
print(f"\n   MTP Losses:")
for i, loss in enumerate(mtp_losses):
    print(f"     Depth {i}: {loss:.2f}")
print(f"   Average MTP Loss:       {avg_mtp_loss:.2f}")
print(f"\n   MTP Weight (λ):         {mtp_weight}")
print(f"\n   Total Loss = {main_loss:.2f} + {mtp_weight} × {avg_mtp_loss:.2f}")
print(f"             = {main_loss:.2f} + {mtp_weight * avg_mtp_loss:.2f}")
print(f"             = {total_loss:.2f}")

print("\n" + "=" * 60)
print("💡 MTP 손실은 보조 손실로, 메인 손실보다 낮은 가중치 적용")


## 5. 혁신 요약

In [ ]:
print("""
🏆 DeepSeek V3 주요 혁신 요약
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1️⃣ Multi-head Latent Attention (MLA)
   ─────────────────────────────────
   핵심: KV 캐시를 저차원 잠재 벡터로 압축
   
   수식: c_KV = W_DKV × h
         k = [W_UK × c_KV; RoPE(W_KR × h)]
   
   효과: ✓ KV 캐시 메모리 73% 절감
         ✓ 긴 컨텍스트 (160K) 처리 가능

2️⃣ DeepSeekMoE with Auxiliary-Loss-Free Load Balancing
   ─────────────────────────────────────────────────────
   핵심: Bias term으로 로드 밸런싱
   
   수식: s'_i = s_i + b_i
         y = Shared(h) + Σ gᵢ × Expertᵢ(h)
   
   효과: ✓ 256개 전문가 중 8개만 활성화
         ✓ 671B 파라미터, 37B 활성화

3️⃣ Multi-Token Prediction (MTP)
   ─────────────────────────────
   핵심: 한 번에 여러 토큰 예측
   
   효과: ✓ 학습 효율성 향상
         ✓ Speculative Decoding으로 추론 가속화

4️⃣ FP8 Mixed Precision Training
   ─────────────────────────────
   핵심: FP8 정밀도로 학습
   
   효과: ✓ 학습 비용 약 $5.6M (매우 저렴!)
         ✓ 성능 저하 없음

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print("📊 성능: 오픈소스 모델 중 최고 성능, GPT-4o/Claude 3.5와 경쟁")

## 📝 최종 퀴즈

In [ ]:
final_quiz = {
    "Q1: DeepSeek V3의 총 파라미터는 671B이다": None,
    "Q2: 토큰당 활성화 파라미터는 37B이다": None,
    "Q3: 모든 61개 레이어가 MoE를 사용한다": None,
    "Q4: MLA는 KV 캐시 메모리를 절감한다": None,
    "Q5: MTP는 추론 속도를 향상시킨다": None,
}

# 여기에 답을 입력하세요 (True or False)

In [ ]:
final_answers = {
    "Q1: DeepSeek V3의 총 파라미터는 671B이다": True,
    "Q2: 토큰당 활성화 파라미터는 37B이다": True,
    "Q3: 모든 61개 레이어가 MoE를 사용한다": False,  # 처음 3개는 Dense
    "Q4: MLA는 KV 캐시 메모리를 절감한다": True,
    "Q5: MTP는 추론 속도를 향상시킨다": True,
}

print("📋 최종 퀴즈 정답")
print("=" * 60)
correct = 0
for q, a in final_answers.items():
    user_ans = final_quiz.get(q)
    is_correct = user_ans == a
    if is_correct:
        correct += 1
    status = "✅" if is_correct else "❌"
    print(f"{status} {q}")
    print(f"   정답: {a}, 당신의 답: {user_ans}\n")

print(f"\n🎯 최종 점수: {correct}/{len(final_answers)} ({correct/len(final_answers)*100:.0f}%)")

## 🎉 학습 완료!

축하합니다! DeepSeek V3 아키텍처 학습을 완료했습니다.

### 📚 학습 내용 요약

1. **01_overview** - 전체 아키텍처 개요
2. **02_rmsnorm_rope** - RMSNorm과 RoPE
3. **03_mla_attention** - Multi-head Latent Attention
4. **04_moe_routing** - DeepSeekMoE와 라우팅
5. **05_full_model** - 전체 모델 구조와 추론 흐름

### 📖 추가 학습 자료

- 논문: https://arxiv.org/pdf/2412.19437
- 코드: https://github.com/huggingface/transformers/blob/main/src/transformers/models/deepseek_v3/
- 모델: https://huggingface.co/deepseek-ai/DeepSeek-V3